In [1]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json
from langchain import hub
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_groq import ChatGroq

In [2]:
# 預處理documents
json_file_path = "example-api-data.json"
with open(json_file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

In [3]:
CHUNK_SIZE = 100
CHUNK_OVERLAP = 10
All_passage_id = {}
original_id_to_new_id = {}
new_id_to_original_id = {}
def process_documents(data):
    documents = []
    now_id = 1
    for comment in data["content"]:
        for fact in comment["facts"]:
            for reference in fact["references"]:
                if(reference["id"] not in original_id_to_new_id):
                    text = reference["title"] + "。" + reference["description"]
                    document = Document(
                        page_content=text,
                        metadata={
                            "original_id": reference["id"],
                            "new_id": now_id,  
                        }
                    )
                    documents.append(document)
                    original_id_to_new_id[reference["id"]] = str(now_id)
                    new_id_to_original_id[str(now_id)] = reference["id"]
                    now_id += 1
              
        
    # 創建text splitter並添加索引追踪
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, 
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    # 分割文檔並添加索引信息
    splits = text_splitter.split_documents(documents)

    return splits

# 有幾個文檔，就有幾個count_for_each_doc

all_splits = process_documents(data)

# 打印處理後的文件，包含索引信息
for split in all_splits:
    print(f"內容: {split.page_content}")
    print(f"索引: {split.metadata}")

內容: 周杰倫演唱會「黃牛票」何時釋出？拓元低調回應 歌迷再度崩潰了 | 娛樂 | NOWnews今日新聞。這篇文章說明了周杰倫演唱會黃牛票的問題，並提到拓元的官方回應。
索引: {'original_id': '3ee75de9-051a-4d04-8b64-14b8e4aa18ac', 'new_id': 1}
內容: 演唱會門票搶不過黃牛⋯日本不比手速、靠運氣的「抽選制」，台灣行得通嗎？－法律白話文運動｜商周。文章討論了日本演唱會使用抽選制以減少黃牛票，並探討其在台灣的適用性。
索引: {'original_id': '556909b4-6a18-4390-bfff-87885d49e285', 'new_id': 2}


In [7]:
print(original_id_to_new_id)

{'3ee75de9-051a-4d04-8b64-14b8e4aa18ac': '1', '556909b4-6a18-4390-bfff-87885d49e285': '2'}


In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector_store = FAISS.from_documents(all_splits, embeddings)

C:\Users\kevin\AppData\Local\Temp\ipykernel_18228\4186356471.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [5]:
# 建立QA系統
# See full prompt at https://smith.langchain.com/hub/langchain-ai/retrieval-qa-chat
import os
from dotenv import load_dotenv

load_dotenv()
my_groq_api_key=os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=my_groq_api_key)
retrieval_qa_chat_prompt = hub.pull("langchain-ai/retrieval-qa-chat")

combine_docs_chain = create_stuff_documents_chain(llm, retrieval_qa_chat_prompt)
rag_chain = create_retrieval_chain(vector_store.as_retriever(), combine_docs_chain)
# predined question
question = ["這個議題有哪些重要人物?", "這個事件的時間軸是?", "這個事件的結果是?", "這個事件的影響是?", "這個事件的意義是?"]
final_result = []
for q in question:
    result = rag_chain.invoke({"input": q})
    final_result.append(result)


d:\reference-system\.venv\lib\site-packages\langsmith\client.py:221: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [6]:
print(final_result)


[{'input': '這個議題有哪些重要人物?', 'context': [Document(metadata={'original_id': '3ee75de9-051a-4d04-8b64-14b8e4aa18ac', 'new_id': 1}, page_content='周杰倫演唱會「黃牛票」何時釋出？拓元低調回應 歌迷再度崩潰了 | 娛樂 | NOWnews今日新聞。這篇文章說明了周杰倫演唱會黃牛票的問題，並提到拓元的官方回應。'), Document(metadata={'original_id': '556909b4-6a18-4390-bfff-87885d49e285', 'new_id': 2}, page_content='演唱會門票搶不過黃牛⋯日本不比手速、靠運氣的「抽選制」，台灣行得通嗎？－法律白話文運動｜商周。文章討論了日本演唱會使用抽選制以減少黃牛票，並探討其在台灣的適用性。')], 'answer': '根據所提供的內容，重要人物是周杰倫，以及他的演唱會主辦單位——拓元。'}, {'input': '這個事件的時間軸是?', 'context': [Document(metadata={'original_id': '556909b4-6a18-4390-bfff-87885d49e285', 'new_id': 2}, page_content='演唱會門票搶不過黃牛⋯日本不比手速、靠運氣的「抽選制」，台灣行得通嗎？－法律白話文運動｜商周。文章討論了日本演唱會使用抽選制以減少黃牛票，並探討其在台灣的適用性。'), Document(metadata={'original_id': '3ee75de9-051a-4d04-8b64-14b8e4aa18ac', 'new_id': 1}, page_content='周杰倫演唱會「黃牛票」何時釋出？拓元低調回應 歌迷再度崩潰了 | 娛樂 | NOWnews今日新聞。這篇文章說明了周杰倫演唱會黃牛票的問題，並提到拓元的官方回應。')], 'answer': '根據提供的內容，時間軸是未知的，因為文章中沒有提到具體的日期或時間段。但根據文章的內容，可以推測這個事件可能發生在周杰倫演唱會的門票販售期間。'}, {'input': '這個事件的結果是?', 'context': [Docum

In [7]:
def format_qa_result(final_result): 
    model_input_json = {}
    for i,qa in enumerate(final_result,1):
        question_key = f"Question_{i}"
        model_input_json[question_key] = {}
        model_input_json[question_key]["Question"] = qa['input']
        model_input_json[question_key]["Answer"] = qa['answer']
        model_input_json[question_key]["cited_passages"] = {}
        for j,context in enumerate(qa['context'],1):
            model_input_json[question_key]["cited_passages"][f"passage{j}"] = {
                #"start_idx": context.metadata['chunk_start_idx'],
                #"end_idx": context.metadata['chunk_end_idx'],
                "source_original_id": context.metadata['original_id'],
                "source_new_id": context.metadata['new_id'],
                #"passage_id": context.metadata['passage_id']
            }
    

    return model_input_json
model_input_json = format_qa_result(final_result)

In [8]:
print(model_input_json)

{'Question_1': {'Question': '這個議題有哪些重要人物?', 'Answer': '根據所提供的內容，重要人物是周杰倫，以及他的演唱會主辦單位——拓元。', 'cited_passages': {'passage1': {'source_original_id': '3ee75de9-051a-4d04-8b64-14b8e4aa18ac', 'source_new_id': 1}, 'passage2': {'source_original_id': '556909b4-6a18-4390-bfff-87885d49e285', 'source_new_id': 2}}}, 'Question_2': {'Question': '這個事件的時間軸是?', 'Answer': '根據提供的內容，時間軸是未知的，因為文章中沒有提到具體的日期或時間段。但根據文章的內容，可以推測這個事件可能發生在周杰倫演唱會的門票販售期間。', 'cited_passages': {'passage1': {'source_original_id': '556909b4-6a18-4390-bfff-87885d49e285', 'source_new_id': 2}, 'passage2': {'source_original_id': '3ee75de9-051a-4d04-8b64-14b8e4aa18ac', 'source_new_id': 1}}}, 'Question_3': {'Question': '這個事件的結果是?', 'Answer': '根據提供的內容，似乎並沒有明確指出事件的結果。文章主要是在討論周杰倫演唱會的黃牛票問題、拓元的回應，以及日本的抽選制對於減少黃牛票的適用性。沒有提供最終的結果或解決方案。', 'cited_passages': {'passage1': {'source_original_id': '3ee75de9-051a-4d04-8b64-14b8e4aa18ac', 'source_new_id': 1}, 'passage2': {'source_original_id': '556909b4-6a18-4390-bfff-87885d49e285', 'source_new_id': 2}}}, 'Que

In [9]:
#把QA問答集變成非結構化問答格式 

model_input_string = ""
for key,value in model_input_json.items():
    model_input_string += f"{key[-1]}. {value['Question']}\n"
    model_input_string += f"答: {value['Answer']}"
    if(model_input_string[-1] == "。"):
        model_input_string = model_input_string[:-1]
    for context in value['cited_passages']:
        #print(context)
        model_input_string += f"[{value['cited_passages'][context]['source_new_id']}]"
    model_input_string += "\n"
print(model_input_string)


1. 這個議題有哪些重要人物?
答: 根據所提供的內容，重要人物是周杰倫，以及他的演唱會主辦單位——拓元[1][2]
2. 這個事件的時間軸是?
答: 根據提供的內容，時間軸是未知的，因為文章中沒有提到具體的日期或時間段。但根據文章的內容，可以推測這個事件可能發生在周杰倫演唱會的門票販售期間[2][1]
3. 這個事件的結果是?
答: 根據提供的內容，似乎並沒有明確指出事件的結果。文章主要是在討論周杰倫演唱會的黃牛票問題、拓元的回應，以及日本的抽選制對於減少黃牛票的適用性。沒有提供最終的結果或解決方案[1][2]
4. 這個事件的影響是?
答: 這個事件的影響是讓歌迷再度崩潰了，因為周杰倫演唱會的黃牛票問題仍然存在，並且對於歌迷來說，購買演唱會門票變得更加困難。另外，這個事件也引發了對於如何減少黃牛票和改善演唱會門票購買體驗的討論和探索，例如採用日本的抽選制[1][2]
5. 這個事件的意義是?
答: 這個事件的意義在於討論演唱會門票的黃牛票問題，以及如何減少黃牛票的影響。文章提到了日本的抽選制，並探討了這種制度在台灣的適用性，意在尋找解決黃牛票問題的方法，讓歌迷能夠公平地購買演唱會門票[1][2]



In [10]:
def load_few_shot(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()
few_shot_example = load_few_shot("few-shot-example.txt")


In [11]:
print(few_shot_example)


###範例1###
QA問答集：

這個事件中的主要人物有誰？
答：小布希總統、財政部長保爾森、聯準會主席柏南克、雷曼兄弟CEO理查德·富爾德[1]
這個事件的時間軸是？
答：2008年9月15日雷曼兄弟宣布破產為關鍵時間點[2][3]
這個事件的經過是？
答：從次貸危機爆發，到雷曼兄弟倒閉，引發全球金融市場動盪[4][5]
這個事件的結果是？
答：美國政府推出7000億美元紓困方案，全球經濟陷入衰退[6][7]
這個事件的影響是？
答：失業率攀升，金融監管改革，全球經濟格局改變[8][9]
這個事件的意義是？
答：促使全球金融體系改革，重新思考金融監管制度[10][11]


摘要回答：
2008年金融危機中，小布希總統、財政部長保爾森、聯準會主席柏南克和雷曼兄弟CEO理查德·富爾德成為關鍵人物[1]。危機於2008年9月15日達到頂點，當天雷曼兄弟宣布破產[2]，隨後引發連鎖反應，造成全球金融市場劇烈動盪[4]。美國政府被迫採取緊急措施，推出規模達7000億美元的紓困方案，試圖穩定金融市場[6]。然而，全球經濟仍陷入嚴重衰退，美國失業率攀升至10%以上，多個金融機構倒閉或被收購[8]。這場危機促使各國政府加強金融監管，建立更嚴格的風險控制機制，徹底改變了全球金融監管格局[10]。其深遠意義在於促使人們重新思考金融體系的脆弱性，推動了全球金融改革，並強化了國際金融合作機制[11]。
###範例1結束###

###範例2開始###
範例2：
QA問答集：

這個事件中的主要人物有誰？
答：世衛組織總幹事譚德塞、各國領導人如習近平、川普等[1]
這個事件的時間軸是？
答：2019年12月首次發現，2020年3月11日被宣布為全球大流行[2]
這個事件的經過是？
答：從中國武漢首次發現病例，到全球蔓延，各國採取防疫措施[3][4]
這個事件的結果是？
答：全球確診破億，各國實施封鎖措施，經濟受創[5][6]
這個事件的影響是？
答：改變生活方式，促進遠距工作，加速數位轉型[7][8]
這個事件的意義是？
答：凸顯全球衛生治理重要性，改變國際合作模式[9][10]


摘要回答：
在這場全球性衛生危機中，世衛組織總幹事譚德塞以及各國領導人如習近平、川普等扮演關鍵角色[1]。疫情於2019年12月在中國武漢首次被發現，到2020年3月11日被世衛組織正式宣布為全球大流行[

In [12]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template(
    '''
    您是一位專業的文章撰寫者，請根據以下的撰寫要求，撰寫一份完整且連貫的摘要報告。
撰寫要求：
1. 內容結構：
   - 按照時間順序或邏輯順序組織內容
   - 確保段落之間的轉折自然
   - 使用適當的連接詞來增加文章流暢度
   - 除了摘要內容之外不要有任何多餘的文字
   - 在輸出的內容中不要有任何除了摘要內容和引用標註之外的描述性文字
   - 不可以用列點的方式來輸出
   - 必須嚴格遵守在結束一個句子之後有兩個換行符號

2. 引用格式：
   - 每個論述都必須標註來源：[n]
   - 如果一個段落包含多個來源的內容，需分別標註

3. 寫作風格：
   - 使用客觀、專業的語氣
   - 避免重複QA中的問題形式
   - 將問答轉化為敘事性的描述

4. 內容完整性：
   - 確保涵蓋所有QA中的重要信息
   - 適當整合相關信息，避免過度分散
   - 在不同觀點間取得平衡

以下是範例問答輸入和範例摘要示例輸出包含一系列的問題和答案：
{examples}

以下是一個你需要處理的問答集，你要對這個問答及做摘要其中包含一系列的問題和答案：
{input}

請以以下格式回答：
###輸出開始###
[你的回答]
###輸出結束###
'''
)

answer_to_abstract_chain = prompt | llm
result = answer_to_abstract_chain.invoke({"input": model_input_string, "examples": few_shot_example})
print(result)




content='###輸出開始###\n周杰倫演唱會的門票販售期間，周杰倫本人以及他的演唱會主辦單位——拓元成為了焦點人物[1][2]。儘管具體的時間軸並未被明確指出，但根據文章的內容，可以推測這個事件可能發生在演唱會門票開始販售的時期[2][1]。這個事件的結果並沒有被明確界定，主要討論集中在周杰倫演唱會的黃牛票問題、拓元的回應，以及日本的抽選制對於減少黃牛票的適用性[1][2]。這個事件對歌迷產生了直接的影響，讓購買演唱會門票變得更加困難，並引發了對於如何減少黃牛票和改善演唱會門票購買體驗的討論和探索，例如採用日本的抽選制[1][2]。此事件的意義在於討論演唱會門票的黃牛票問題，以及如何減少黃牛票的影響，文章提到了日本的抽選制，並探討了這種制度在台灣的適用性，意在尋找解決黃牛票問題的方法，讓歌迷能夠公平地購買演唱會門票[1][2]。\n\n \n###輸出結束###' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 317, 'prompt_tokens': 1847, 'total_tokens': 2164, 'completion_time': 1.152727273, 'prompt_time': 0.201493182, 'queue_time': 0.068274202, 'total_time': 1.354220455}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_fcc3b74982', 'finish_reason': 'stop', 'logprobs': None} id='run-6aace9db-1787-402d-af6c-46cf80819aac-0' usage_metadata={'input_tokens': 1847, 'output_tokens': 317, 'total_tokens': 2164}


In [18]:
print(result.content)

###輸出開始###
周杰倫演唱會的門票販售期間，周杰倫本人以及他的演唱會主辦單位——拓元成為了焦點人物[1][2]。儘管具體的時間軸並未被明確指出，但根據文章的內容，可以推測這個事件可能發生在演唱會門票開始販售的時期[2][1]。這個事件的結果並沒有被明確界定，主要討論集中在周杰倫演唱會的黃牛票問題、拓元的回應，以及日本的抽選制對於減少黃牛票的適用性[1][2]。這個事件對歌迷產生了直接的影響，讓購買演唱會門票變得更加困難，並引發了對於如何減少黃牛票和改善演唱會門票購買體驗的討論和探索，例如採用日本的抽選制[1][2]。此事件的意義在於討論演唱會門票的黃牛票問題，以及如何減少黃牛票的影響，文章提到了日本的抽選制，並探討了這種制度在台灣的適用性，意在尋找解決黃牛票問題的方法，讓歌迷能夠公平地購買演唱會門票[1][2]。

 
###輸出結束###


In [19]:

result_cut_front_and_end=result.content[11:-11]
print(result_cut_front_and_end)

周杰倫演唱會的門票販售期間，周杰倫本人以及他的演唱會主辦單位——拓元成為了焦點人物[1][2]。儘管具體的時間軸並未被明確指出，但根據文章的內容，可以推測這個事件可能發生在演唱會門票開始販售的時期[2][1]。這個事件的結果並沒有被明確界定，主要討論集中在周杰倫演唱會的黃牛票問題、拓元的回應，以及日本的抽選制對於減少黃牛票的適用性[1][2]。這個事件對歌迷產生了直接的影響，讓購買演唱會門票變得更加困難，並引發了對於如何減少黃牛票和改善演唱會門票購買體驗的討論和探索，例如採用日本的抽選制[1][2]。此事件的意義在於討論演唱會門票的黃牛票問題，以及如何減少黃牛票的影響，文章提到了日本的抽選制，並探討了這種制度在台灣的適用性，意在尋找解決黃牛票問題的方法，讓歌迷能夠公平地購買演唱會門票[1][2]。

 


In [20]:
import re
# 製作最後的json file輸出
final_output_format={}
# 解構result有引用到passage_id
citations=re.findall(r'\[(\d+)\]',result_cut_front_and_end)

unique_citations=sorted(set(map(int,citations)))
print(unique_citations)
final_output_format["Summary"]=result_cut_front_and_end
final_output_format["Citations"]={}
for citation in unique_citations:
    final_output_format["Citations"][str(citation)]=new_id_to_original_id[str(citation)]
print(final_output_format)


[1, 2]
{'Summary': '周杰倫演唱會的門票販售期間，周杰倫本人以及他的演唱會主辦單位——拓元成為了焦點人物[1][2]。儘管具體的時間軸並未被明確指出，但根據文章的內容，可以推測這個事件可能發生在演唱會門票開始販售的時期[2][1]。這個事件的結果並沒有被明確界定，主要討論集中在周杰倫演唱會的黃牛票問題、拓元的回應，以及日本的抽選制對於減少黃牛票的適用性[1][2]。這個事件對歌迷產生了直接的影響，讓購買演唱會門票變得更加困難，並引發了對於如何減少黃牛票和改善演唱會門票購買體驗的討論和探索，例如採用日本的抽選制[1][2]。此事件的意義在於討論演唱會門票的黃牛票問題，以及如何減少黃牛票的影響，文章提到了日本的抽選制，並探討了這種制度在台灣的適用性，意在尋找解決黃牛票問題的方法，讓歌迷能夠公平地購買演唱會門票[1][2]。\n\n ', 'Citations': {'1': '3ee75de9-051a-4d04-8b64-14b8e4aa18ac', '2': '556909b4-6a18-4390-bfff-87885d49e285'}}


In [21]:
import json

# 将final_output_format转换为JSON并保存
with open('summary_output.json', 'w', encoding='utf-8') as f:
    json.dump(final_output_format, f, ensure_ascii=False, indent=4)